In [20]:
import matplotlib.pyplot as plt 
import pandas as pd
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages


In [21]:
data = pd.read_csv('data/intresting_data_to_visualize/cancer_b.csv',header=0)
numdata = data.select_dtypes(include=['int64','float64'])
numdata = numdata.iloc[:,1:]
numdata

,Radius (mean),Texture (mean),Perimeter (mean),Area (mean),Smoothness (mean),Compactness (mean),Concavity (mean),Concave points (mean),Symmetry (mean),Fractal dimension (mean),...,Radius (worst),Texture (worst),Perimeter (worst),Area (worst),Smoothness (worst),Compactness (worst),Concavity (worst),Concave points (worst),Symmetry (worst),Fractal dimension (worst)
0,13.540,14.36,87.46,566.3,0.09779,0.08129,0.06664,0.047810,0.1885,0.05766,...,15.110,19.26,99.70,711.2,0.14400,0.17730,0.23900,0.12880,0.2977,0.07259
1,13.080,15.71,85.63,520.0,0.10750,0.12700,0.04568,0.031100,0.1967,0.06811,...,14.500,20.49,96.09,630.5,0.13120,0.27760,0.18900,0.07283,0.3184,0.08183
2,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.020760,0.1815,0.06905,...,10.230,15.66,65.13,314.9,0.13240,0.11480,0.08867,0.06227,0.2450,0.07773
3,13.030,18.42,82.61,523.8,0.08983,0.03766,0.02562,0.029230,0.1467,0.05863,...,13.300,22.81,84.46,545.9,0.09701,0.04619,0.04833,0.05013,0.1987,0.06169
4,8.196,16.84,51.71,201.9,0.08600,0.05943,0.01588,0.005917,0.1769,0.06503,...,8.964,21.96,57.26,242.2,0.12970,0.13570,0.06880,0.02564,0.3105,0.07409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
352,14.590,22.68,96.39,657.1,0.08473,0.13300,0.10290,0.037360,0.1454,0.06147,...,15.480,27.27,105.90,733.5,0.10260,0.31710,0.36620,0.11050,0.2258,0.08004
353,11.510,23.93,74.52,403.5,0.09261,0.10210,0.11120,0.041050,0.1388,0.06570,...,12.480,37.16,82.28,474.2,0.12980,0.25170,0.36300,0.09653,0.2112,0.08732
354,14.050,27.15,91.38,600.4,0.09929,0.11260,0.04462,0.043040,0.1537,0.06171,...,15.300,33.17,100.20,706.7,0.12410,0.22640,0.13260,0.10480,0.2250,0.08321
355,11.200,29.37,70.67,386.0,0.07449,0.03558,0.00000,0.000000,0.1060,0.05502,...,11.920,38.30,75.19,439.6,0.09267,0.05494,0.00000,0.00000,0.1566,0.05905


In [23]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

bins = 10
output_dir = "./all_text_files/"
os.makedirs(output_dir, exist_ok=True)

with PdfPages(os.path.join(output_dir, "cancerColumens.pdf")) as pdf:
    for i in range(len(numdata.columns)):
        column_name = numdata.columns[i]
        current_column = numdata.iloc[:, i].values
    
        plt.figure(figsize=(10, 7))
        plt.title(f'Column {i+1}: {column_name}')
        plt.hist(current_column, bins=bins, edgecolor='black')
        plt.ylabel('Frequency')
        plt.xlabel(column_name)
        pdf.savefig()  
        plt.close()

        txt_lines = []
        txt_lines.append(f"=== Column {i+1}: {column_name} ===\n")
        txt_lines.append(f"Total Data Points: {len(current_column)}\n")

        max_value = current_column.max()
        min_value = current_column.min()
        total_count = len(current_column)
        start = min_value
        binrange = round((max_value - min_value) / bins, 2)

        subgraph_pdf_path = os.path.join(output_dir, f"subgraph{i+1}.pdf")
        subgraph_txt_path = os.path.join(output_dir, f"subgraph{i+1}.txt")

        with PdfPages(subgraph_pdf_path) as pdfbin:
            print(f"Generating {subgraph_pdf_path} and {subgraph_txt_path}")

            for j in range(bins):
                offset = round(start + binrange, 2)
                new_numdata = current_column[(current_column >= start) & (current_column <= offset)]

                txt_lines.append(f"\n--- Histogram of Bin {j+1} ---\n")
                txt_lines.append(f"Range: {start:.4f} - {offset:.4f}\n")

                interpret_data = np.array(new_numdata)
                if len(interpret_data) > 0:
                    min_interpret = interpret_data.min()
                    max_interpret = interpret_data.max()
                    bin_width = (max_interpret - min_interpret) / bins if bins > 0 else 0

                    
                    for k in range(bins):
                        start_range = min_interpret + bin_width * k
                        end_range = start_range + bin_width
                        if k == bins - 1:
                            end_range = max_interpret

                        bin_data = interpret_data[(interpret_data >= start_range) & (interpret_data <= end_range)]
                        count = len(bin_data)

                        if len(new_numdata) > 0:
                            percent = (count / len(new_numdata)) * 100
                            if percent==100:
                                txt_lines.append(f"{start_range:.4f} - {end_range:.4f} -> {percent:.2f}%\n")
                                break
                            if percent!=0:
                                txt_lines.append(f"Bin {k+1}: {start_range:.4f} - {end_range:.4f} -> {percent:.2f}%\n")
                
                plt.figure(figsize=(10, 6))
                plt.hist(new_numdata, bins=bins, edgecolor='black')
                plt.title(f'Column {i+1} - Bin {j+1}')
                plt.xlabel(column_name)
                plt.ylabel('Frequency')
                pdfbin.savefig()
                plt.close()

                start = offset  

        with open(subgraph_txt_path, 'w') as f:
            f.writelines(txt_lines)

print("end")

Generating ./all_text_files/subgraph1.pdf and ./all_text_files/subgraph1.txt
Generating ./all_text_files/subgraph2.pdf and ./all_text_files/subgraph2.txt
Generating ./all_text_files/subgraph3.pdf and ./all_text_files/subgraph3.txt
Generating ./all_text_files/subgraph4.pdf and ./all_text_files/subgraph4.txt
Generating ./all_text_files/subgraph5.pdf and ./all_text_files/subgraph5.txt
Generating ./all_text_files/subgraph6.pdf and ./all_text_files/subgraph6.txt
Generating ./all_text_files/subgraph7.pdf and ./all_text_files/subgraph7.txt
Generating ./all_text_files/subgraph8.pdf and ./all_text_files/subgraph8.txt
Generating ./all_text_files/subgraph9.pdf and ./all_text_files/subgraph9.txt
Generating ./all_text_files/subgraph10.pdf and ./all_text_files/subgraph10.txt
Generating ./all_text_files/subgraph11.pdf and ./all_text_files/subgraph11.txt
Generating ./all_text_files/subgraph12.pdf and ./all_text_files/subgraph12.txt
Generating ./all_text_files/subgraph13.pdf and ./all_text_files/subgra